In [1]:
!pip install -q transformers accelerate bitsandbytes datasets netcal scikit-learn wandb

In [2]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sagarsharma-ai (sagarsharma-ai-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import torch
import numpy as np
import wandb
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from netcal.metrics import ECE
from sklearn.metrics import brier_score_loss

wandb.init(
    project="Order_of_Operations",
    name="pipeline_a_eval",
    config={
        "model": "google/gemma-2-2b-it",
        "quantization": "NF4_4bit",
        "pipeline": "A_PTQ",
        "eval_datasets": ["mmlu", "arc_challenge", "truthfulqa"],
        "n_samples": 300
    }
)

### Gemma-2-2B-IT

In [5]:
from huggingface_hub import login
login()

model_id = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [6]:
answer_tokens = ["A", "B", "C", "D"]
answer_token_ids = [tokenizer.encode(t, add_special_tokens=False)[0] for t in answer_tokens]
print(dict(zip(answer_tokens, answer_token_ids)))

{'A': 235280, 'B': 235305, 'C': 235288, 'D': 235299}


In [12]:
def get_answer_probs(prompt_text, model, tokenizer, answer_token_ids):
    messages = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    last_logits = outputs.logits[0, -1, :].float()  # cast bf16 → fp32
    answer_logits = last_logits[answer_token_ids]
    probs = torch.softmax(answer_logits, dim=-1).cpu().numpy()

    return probs

In [13]:
def load_mmlu(n=300):
    ds = load_dataset("cais/mmlu", "all", split="test")
    return ds.shuffle(seed=42).select(range(n))

def load_arc(n=300):
    ds = load_dataset("ai2_arc", "ARC-Challenge", split="test")
    return ds.shuffle(seed=42).select(range(n))

def load_truthfulqa(n=300):
    ds = load_dataset("truthful_qa", "multiple_choice", split="validation")
    return ds.shuffle(seed=42).select(range(n))

def format_mmlu(example):
    choices = example["choices"]
    prompt = (
        f"Answer the following multiple choice question. "
        f"Reply with only the letter A, B, C, or D.\n\n"
        f"Question: {example['question']}\n"
        f"A) {choices[0]}\nB) {choices[1]}\nC) {choices[2]}\nD) {choices[3]}"
    )
    return prompt, example["answer"]

def format_arc(example):
    labels_map = {"A": 0, "B": 1, "C": 2, "D": 3, "1": 0, "2": 1, "3": 2, "4": 3}
    choices = example["choices"]["text"]
    while len(choices) < 4:
        choices.append("N/A")
    prompt = (
        f"Answer the following multiple choice question. "
        f"Reply with only the letter A, B, C, or D.\n\n"
        f"Question: {example['question']}\n"
        f"A) {choices[0]}\nB) {choices[1]}\nC) {choices[2]}\nD) {choices[3]}"
    )
    return prompt, labels_map.get(example["answerKey"], 0)

def format_truthfulqa(example):
    choices = example["mc1_targets"]["choices"][:4]
    while len(choices) < 4:
        choices.append("N/A")
    correct_idx = example["mc1_targets"]["labels"].index(1)
    if correct_idx > 3:
        correct_idx = 0
    prompt = (
        f"Answer the following multiple choice question. "
        f"Reply with only the letter A, B, C, or D.\n\n"
        f"Question: {example['question']}\n"
        f"A) {choices[0]}\nB) {choices[1]}\nC) {choices[2]}\nD) {choices[3]}"
    )
    return prompt, correct_idx

In [14]:
def evaluate_pipeline(model, tokenizer, dataset, format_fn, dataset_name, answer_token_ids):
    results = []

    for i, example in enumerate(dataset):
        try:
            prompt, true_label = format_fn(example)
            probs = get_answer_probs(prompt, model, tokenizer, answer_token_ids)

            pred_label = int(np.argmax(probs))
            confidence = float(probs[pred_label])
            is_correct = int(pred_label == true_label)
            true_prob = float(probs[true_label])

            results.append({
                "dataset": dataset_name,
                "true_label": true_label,
                "pred_label": pred_label,
                "confidence": confidence,
                "true_prob": true_prob,
                "is_correct": is_correct,
                "probs": probs.tolist()
            })

            if i % 50 == 0:
                acc = np.mean([r["is_correct"] for r in results])
                print(f"{dataset_name}: {i}/{len(dataset)} | acc: {acc:.3f}")

        except Exception as e:
            print(f"Skipped {i}: {e}")
            continue

    return results

mmlu_data = load_mmlu(300)
arc_data = load_arc(300)
tqa_data = load_truthfulqa(300)

results_mmlu = evaluate_pipeline(model, tokenizer, mmlu_data, format_mmlu, "MMLU", answer_token_ids)
results_arc = evaluate_pipeline(model, tokenizer, arc_data, format_arc, "ARC", answer_token_ids)
results_tqa = evaluate_pipeline(model, tokenizer, tqa_data, format_truthfulqa, "TruthfulQA", answer_token_ids)

all_results = results_mmlu + results_arc + results_tqa
print(f"Total: {len(all_results)}")

MMLU: 0/300 | acc: 0.000
MMLU: 50/300 | acc: 0.490
MMLU: 100/300 | acc: 0.525
MMLU: 150/300 | acc: 0.517
MMLU: 200/300 | acc: 0.507
MMLU: 250/300 | acc: 0.514
ARC: 0/300 | acc: 1.000
ARC: 50/300 | acc: 0.725
ARC: 100/300 | acc: 0.752
ARC: 150/300 | acc: 0.755
ARC: 200/300 | acc: 0.751
ARC: 250/300 | acc: 0.769
TruthfulQA: 0/300 | acc: 0.000
TruthfulQA: 50/300 | acc: 0.490
TruthfulQA: 100/300 | acc: 0.505
TruthfulQA: 150/300 | acc: 0.497
TruthfulQA: 200/300 | acc: 0.542
TruthfulQA: 250/300 | acc: 0.578
Total: 900


In [15]:
def compute_metrics(results, pipeline_name):
    confidences = np.array([r["confidence"] for r in results])
    true_probs = np.array([r["true_prob"] for r in results])
    is_correct = np.array([r["is_correct"] for r in results])

    ece = ECE(bins=15).measure(confidences, is_correct)
    brier = brier_score_loss(is_correct, confidences)
    nll = -np.mean(np.log(true_probs + 1e-10))
    accuracy = np.mean(is_correct)
    conf_correct = np.mean(confidences[is_correct == 1])
    conf_incorrect = np.mean(confidences[is_correct == 0])
    overconf_rate = np.mean((confidences > 0.7) & (is_correct == 0))
    underconf_rate = np.mean((confidences < 0.5) & (is_correct == 1))

    metrics = {
        "pipeline": pipeline_name,
        "accuracy": accuracy,
        "ECE": ece,
        "brier_score": brier,
        "NLL": nll,
        "conf_when_correct": conf_correct,
        "conf_when_incorrect": conf_incorrect,
        "overconfidence_rate": overconf_rate,
        "underconfidence_rate": underconf_rate,
    }

    for k, v in metrics.items():
        if k != "pipeline":
            print(f"  {k}: {v:.4f}")

    wandb.log({f"pipeline_a/{k}": v for k, v in metrics.items() if k != "pipeline"})
    return metrics

print("=== Pipeline A (PTQ) ===")
metrics_a = compute_metrics(all_results, "Pipeline_A_PTQ")

=== Pipeline A (PTQ) ===
  accuracy: 0.6122
  ECE: 0.2928
  brier_score: 0.2987
  NLL: 1.7259
  conf_when_correct: 0.9433
  conf_when_incorrect: 0.8431
  overconfidence_rate: 0.2967
  underconfidence_rate: 0.0133


In [16]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

df = pd.DataFrame(all_results)
df.to_csv("/content/drive/MyDrive/pipeline_a_results_custom.csv", index=False)

artifact = wandb.Artifact("pipeline_a_results_custom", type="dataset")
artifact.add_file("/content/drive/MyDrive/pipeline_a_results_custom.csv")
wandb.log_artifact(artifact)

wandb.finish()
print("Done.")

Mounted at /content/drive


pipeline_a/ECE,▁
pipeline_a/NLL,▁
pipeline_a/accuracy,▁
pipeline_a/brier_score,▁
pipeline_a/conf_when_correct,▁
pipeline_a/conf_when_incorrect,▁
pipeline_a/overconfidence_rate,▁
pipeline_a/underconfidence_rate,▁
pipeline_a/ECE,0.29285
pipeline_a/NLL,1.72592
pipeline_a/accuracy,0.61222


Done.
